# Ejercicio 3. Consultas directas sobre archivos Parquet

Todas las consultas leen los archivos de `data/raw` con `read_parquet`, `parquet_file_metadata` y `parquet_schema`, sin importar nada a una tabla. El SQL vive en `sql/01_exploracion.sql` y cada celda lo carga por nombre, así que lo que se ve impreso es exactamente lo que se ejecutó.

Este notebook se ejecutó con los datos de 2026 (enero a agosto), antes de incorporar 2024 y 2025.

In [1]:
import sys
sys.path.append("../scripts")

import pandas as pd
from consultas import cargar_consultas, conectar, ejecutar

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

con = conectar(vistas=False)
q = cargar_consultas("01_exploracion.sql")

## 3.1 Cantidad de archivos

**Objetivo.** Saber cuántos archivos hay por tipo de taxi y año, y cuánto pesan.

**Fuente.** Metadatos de `data/raw/*/*/*.parquet` mediante `parquet_file_metadata`, que lee solo el pie de cada archivo.

In [2]:
ejecutar(con, q, "archivos_disponibles")

SELECT
    split_part(file_name, '/', 3) AS tipo_taxi,
    split_part(file_name, '/', 4) AS anio,
    count(*) AS archivos,
    sum(num_rows) AS registros,
    round(sum(file_size_bytes) / 1024 / 1024, 1) AS tamanio_mib
FROM parquet_file_metadata('data/raw/*/*/*.parquet')
GROUP BY ALL
ORDER BY tipo_taxi, anio;

[archivos_disponibles] 2 filas en 0.03 s


,tipo_taxi,anio,archivos,registros,tamanio_mib
0,green,2026,8,337114.0,7.9
1,yellow,2026,8,29703355.0,487.8


**Resultado.** Hay 16 archivos, 8 de taxis amarillos y 8 de verdes, que corresponden a enero a agosto de 2026. Los amarillos ocupan 487.8 MiB y los verdes 7.9 MiB.

**Decisión.** El volumen de los verdes es 60 veces menor, así que cualquier comparación entre tipos debe hacerse con proporciones o promedios y no con totales.

In [3]:
ejecutar(con, q, "registros_por_archivo")

SELECT
    split_part(file_name, '/', 5) AS archivo,
    num_rows AS registros,
    num_row_groups AS grupos_de_filas
FROM parquet_file_metadata('data/raw/*/*/*.parquet')
ORDER BY archivo;

[registros_por_archivo] 16 filas en 0.00 s


,archivo,registros,grupos_de_filas
0,green_tripdata_2026-01.parquet,40272,1
1,green_tripdata_2026-02.parquet,37373,1
2,green_tripdata_2026-03.parquet,44208,1
3,green_tripdata_2026-04.parquet,44238,1
4,green_tripdata_2026-05.parquet,44921,1
5,green_tripdata_2026-06.parquet,44163,1
6,green_tripdata_2026-07.parquet,41252,1
7,green_tripdata_2026-08.parquet,40687,1
8,yellow_tripdata_2026-01.parquet,3724889,4
9,yellow_tripdata_2026-02.parquet,3399866,4


## 3.2 Cantidad de registros

**Objetivo.** Contar los viajes disponibles. Se hace de dos formas para validar: sumando `num_rows` de los metadatos (arriba) y con `count(*)` sobre los datos.

**Fuente.** `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet`.

In [4]:
ejecutar(con, q, "registros_totales")

SELECT 'yellow' AS tipo_taxi, count(*) AS registros
FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true)
UNION ALL
SELECT 'green', count(*)
FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true);

[registros_totales] 2 filas en 0.01 s


,tipo_taxi,registros
0,yellow,29703355
1,green,337114


**Resultado.** Hay 30,040,469 registros, 29,703,355 amarillos y 337,114 verdes. Ambos conteos coinciden con la suma de `num_rows` de los metadatos.

**Decisión.** Como el conteo por metadatos es instantáneo y coincide, se usa para validar descargas futuras sin leer los datos.

## 3.3 y 3.4 Columnas y tipos de datos

**Objetivo.** Conocer el esquema que DuckDB infiere al leer todos los archivos juntos y revisar si el esquema físico cambia entre archivos.

**Fuente.** `DESCRIBE` sobre `read_parquet(..., union_by_name = true)` y `parquet_schema` sobre cada archivo.

In [5]:
ejecutar(con, q, "columnas_yellow")

DESCRIBE SELECT * FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true);

[columnas_yellow] 21 filas en 0.00 s


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [6]:
ejecutar(con, q, "columnas_green")

DESCRIBE SELECT * FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true);

[columnas_green] 22 filas en 0.00 s


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,lpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,lpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,store_and_fwd_flag,VARCHAR,YES,None,None,None
4,RatecodeID,BIGINT,YES,None,None,None
5,PULocationID,INTEGER,YES,None,None,None
6,DOLocationID,INTEGER,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None
8,trip_distance,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


In [7]:
ejecutar(con, q, "columnas_por_archivo")

SELECT
    split_part(file_name, '/', 3) AS tipo_taxi,
    name AS columna,
    string_agg(DISTINCT coalesce(logical_type::VARCHAR, type), ', ') AS tipos_fisicos,
    count(DISTINCT file_name) AS archivos_con_columna
FROM parquet_schema('data/raw/*/*/*.parquet')
WHERE name NOT IN ('schema', 'duckdb_schema')
GROUP BY ALL
ORDER BY tipo_taxi, columna;

[columnas_por_archivo] 43 filas en 0.02 s


,tipo_taxi,columna,tipos_fisicos,archivos_con_columna
0,green,DOLocationID,INT32,8
1,green,PULocationID,INT32,8
2,green,RatecodeID,INT64,8
3,green,VendorID,INT32,8
4,green,cbd_congestion_fee,DOUBLE,8
5,green,congestion_surcharge,DOUBLE,8
6,green,ehail_fee,DOUBLE,8
7,green,extra,DOUBLE,8
8,green,fare_amount,DOUBLE,8
9,green,improvement_surcharge,DOUBLE,8


**Resultado.** Los amarillos tienen 21 columnas y los verdes 22. Las fechas son `TIMESTAMP`, los identificadores (`VendorID`, `PULocationID`, `DOLocationID`) son `INTEGER`, los códigos (`RatecodeID`, `payment_type`, `passenger_count`, `trip_type`) son `BIGINT` y todos los montos son `DOUBLE`.

**Diferencias entre tipos.** Las fechas se llaman `tpep_*` en amarillos y `lpep_*` en verdes. Solo los amarillos tienen `Airport_fee` y solo los verdes tienen `ehail_fee` y `trip_type`.

**Columna nueva.** `request_source` aparece solo en 3 de los 8 archivos de cada tipo. Sin `union_by_name = true` la lectura conjunta fallaría o desalinearía columnas.

**Decisión.** Se usará siempre `union_by_name = true` y se creará una vista `viajes` que renombre las fechas a `inicio` y `fin` para consultar ambos tipos con el mismo SQL (ver `sql/00_vistas.sql`).

## 3.5 Muestra de registros

**Objetivo.** Ver cómo lucen los datos reales. Se usa muestreo con semilla fija para que la muestra sea reproducible.

In [8]:
ejecutar(con, q, "muestra_yellow")

SELECT *
FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true)
USING SAMPLE reservoir(5 ROWS) REPEATABLE (42);



[muestra_yellow] 5 filas en 0.12 s


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,1,2026-01-01 00:34:45,2026-01-01 00:40:15,4,0.90,1,N,262,141,1,7.9,3.50,0.5,3.20,0.00,1.0,16.10,2.5,0.0,0.00,None
1,1,2026-01-01 00:06:07,2026-01-01 00:20:20,2,1.90,1,N,231,90,1,14.2,4.25,0.5,3.95,0.00,1.0,23.90,2.5,0.0,0.75,None
2,2,2026-01-01 03:15:42,2026-01-01 03:20:33,1,1.15,1,N,263,140,1,7.2,1.00,0.5,2.44,0.00,1.0,14.64,2.5,0.0,0.00,None
3,2,2026-01-01 04:25:35,2026-01-01 04:38:18,1,3.69,1,N,107,143,1,17.7,1.00,0.5,4.69,0.00,1.0,28.14,2.5,0.0,0.75,None
4,2,2026-01-01 06:33:56,2026-01-01 06:55:37,2,10.27,2,N,68,138,2,70.0,0.00,0.5,0.00,6.94,1.0,81.69,2.5,0.0,0.75,None


In [9]:
ejecutar(con, q, "muestra_green")

SELECT *
FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true)
USING SAMPLE reservoir(5 ROWS) REPEATABLE (42);



[muestra_green] 5 filas en 0.01 s


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,request_source
0,2,2026-01-02 05:43:34,2026-01-02 05:44:09,N,5,34,34,2,0.40,70.0,0.0,0.0,14.20,0.0,NaN,1.0,85.20,1,2,0.0,0.0,None
1,2,2026-01-02 21:02:00,2026-01-02 21:07:42,N,1,74,41,1,1.17,7.9,1.0,0.5,2.08,0.0,NaN,1.0,12.48,1,1,0.0,0.0,None
2,1,2026-01-14 21:03:03,2026-01-14 21:06:16,N,1,181,65,1,0.60,5.8,1.0,1.5,0.00,0.0,NaN,1.0,8.30,2,1,0.0,0.0,None
3,2,2026-01-16 19:11:20,2026-01-16 19:29:44,N,1,74,151,1,2.33,17.7,2.5,0.5,4.34,0.0,NaN,1.0,26.04,1,1,0.0,0.0,None
4,2,2026-01-19 02:15:45,2026-01-19 02:20:57,N,1,95,102,1,1.35,8.6,1.0,0.5,1.00,0.0,NaN,1.0,12.10,1,1,0.0,0.0,None


**Resultado.** La mayoría de filas son viajes cortos dentro de Manhattan con tarifa de taxímetro (`RatecodeID = 1`) y pago con tarjeta (`payment_type = 1`). Ya aparece `cbd_congestion_fee` de 0.75, el cargo por congestión de Manhattan. Un viaje verde de 0.4 millas cobra 70 dólares con `RatecodeID = 5` (tarifa negociada) y `trip_type = 2` (despacho), lo que muestra que la distancia sola no explica el monto.

**Decisión.** El análisis de tarifas debe separar por `RatecodeID` y `trip_type`, porque los viajes negociados distorsionan los promedios.

## 3.6 Problemas de calidad de datos

**Objetivo.** Medir de forma sistemática los problemas visibles en la exploración.

**Fuente.** `SUMMARIZE` sobre cada tipo y una consulta con conteos filtrados sobre ambos.

In [10]:
ejecutar(con, q, "resumen_yellow", mostrar_sql=False)

[resumen_yellow] 21 filas en 15.73 s


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,7,4,1.885739809526567,0.7155277514022337,2,2,2,29703355,0.00
1,tpep_pickup_datetime,TIMESTAMP,2001-01-01 09:23:58,2026-08-31 23:59:59,16867928,2026-04-30 15:29:29.519432,NaN,2026-03-04 06:37:15.910762,2026-04-30 14:33:23.357605,2026-06-26 09:21:24.376303,29703355,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2001-01-01 16:09:38,2026-09-01 20:16:00,16587125,2026-04-30 15:47:08.186216,NaN,2026-03-04 07:05:14.345412,2026-04-30 18:44:59.211607,2026-06-26 00:39:57.59215,29703355,0.00
3,passenger_count,BIGINT,0,9,11,1.2494318488564,0.6529195072996752,1,1,1,29703355,25.98
4,trip_distance,DOUBLE,0.0,328522.2,7217,5.552940149690039,550.6497893231032,1.0236246502594042,1.8547796064460642,3.8076978698852217,29703355,0.00
5,RatecodeID,BIGINT,1,99,7,4.527471444398553,18.000926540067038,1,1,1,29703355,25.98
6,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,29703355,25.98
7,PULocationID,INTEGER,1,265,290,161.57793013617484,66.74656703251728,117,161,233,29703355,0.00
8,DOLocationID,INTEGER,1,265,298,161.05139342003622,70.72548569211288,109,162,234,29703355,0.00
9,payment_type,BIGINT,0,5,6,0.8621735154160195,0.6463324345961725,0,1,1,29703355,0.00


In [11]:
ejecutar(con, q, "resumen_green", mostrar_sql=False)

[resumen_green] 22 filas en 0.26 s


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,6,3,2.328351833504393,1.2771882973738604,2,2,2,337114,0.00
1,lpep_pickup_datetime,TIMESTAMP,2008-12-31 17:35:31,2026-08-31 23:58:28,332702,2026-05-02 13:25:51.096427,NaN,2026-03-06 03:58:08.522032,2026-05-04 10:41:36.943872,2026-06-28 01:56:08.640136,337114,0.00
2,lpep_dropoff_datetime,TIMESTAMP,2008-12-31 23:16:26,2026-09-02 09:39:37,396286,2026-05-02 13:46:38.833759,NaN,2026-03-06 04:03:11.021048,2026-05-02 23:58:39.035715,2026-06-28 14:52:30.326917,337114,0.00
3,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,337114,14.47
4,RatecodeID,BIGINT,1,99,7,1.2549672434183374,1.0015327424296085,1,1,1,337114,14.47
5,PULocationID,INTEGER,1,265,263,97.32761024460568,56.57830275002365,74,75,104,337114,0.00
6,DOLocationID,INTEGER,1,265,266,142.8768458147689,77.24476821098119,75,140,229,337114,0.00
7,passenger_count,BIGINT,0,9,11,1.3006461144694266,0.9481007387491747,1,1,1,337114,14.47
8,trip_distance,DOUBLE,0.0,179830.92,2472,13.349507644298376,880.4035093234568,1.2538149329443218,2.06821956058534,3.693803054489612,337114,0.00
9,fare_amount,DOUBLE,-500.0,1676.7,4808,17.014101995170652,17.958627635270645,8.603242769204591,13.280197621168186,19.660143166840136,337114,0.00


In [12]:
ejecutar(con, q, "problemas_calidad")

WITH base AS (
    SELECT 'yellow' AS tipo_taxi, tpep_pickup_datetime AS inicio, tpep_dropoff_datetime AS fin,
           passenger_count, trip_distance, fare_amount, total_amount, RatecodeID, payment_type,
           filename
    FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true, filename = true)
    UNION ALL
    SELECT 'green', lpep_pickup_datetime, lpep_dropoff_datetime,
           passenger_count, trip_distance, fare_amount, total_amount, RatecodeID, payment_type,
           filename
    FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true, filename = true)
)
SELECT
    tipo_taxi,
    count(*) AS registros,
    count(*) FILTER (WHERE passenger_count IS NULL) AS pasajeros_nulos,
    count(*) FILTER (WHERE passenger_count = 0) AS pasajeros_cero,
    count(*) FILTER (WHERE trip_distance = 0) AS distancia_cero,
    count(*) FILTER (WHERE trip_distance > 100) AS distancia_mayor_100mi,
    count(*) FILTER (WHERE fare_amount < 0) AS tarifa_negativa,
  

[problemas_calidad] 2 filas en 0.76 s


,tipo_taxi,registros,pasajeros_nulos,pasajeros_cero,distancia_cero,distancia_mayor_100mi,tarifa_negativa,total_negativo,fin_antes_de_inicio,duracion_mayor_24h,fecha_fuera_del_mes_del_archivo,ratecode_desconocido,pago_tipo_cero
0,green,337114,48775,4527,12212,72,999,1023,234,4,98,48777,0
1,yellow,29703355,7716688,91359,952231,1223,157364,161835,371683,254,146,8486381,7716688


In [13]:
ejecutar(con, q, "fechas_fuera_de_rango")

SELECT
    'yellow' AS tipo_taxi,
    min(tpep_pickup_datetime) AS inicio_minimo,
    max(tpep_pickup_datetime) AS inicio_maximo
FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true)
UNION ALL
SELECT 'green', min(lpep_pickup_datetime), max(lpep_pickup_datetime)
FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true);

[fechas_fuera_de_rango] 2 filas en 0.11 s


,tipo_taxi,inicio_minimo,inicio_maximo
0,yellow,2001-01-01 09:23:58,2026-08-31 23:59:59
1,green,2008-12-31 17:35:31,2026-08-31 23:58:28


In [14]:
ejecutar(con, q, "nulos_por_archivo")

SELECT
    regexp_extract(filename, '([a-z]+_tripdata_\d{4}-\d{2})', 1) AS archivo,
    count(*) AS registros,
    round(100.0 * count(*) FILTER (WHERE passenger_count IS NULL) / count(*), 2) AS pct_pasajeros_nulos
FROM read_parquet('data/raw/*/*/*.parquet', union_by_name = true, filename = true)
GROUP BY archivo
ORDER BY archivo;

[nulos_por_archivo] 16 filas en 0.05 s


,archivo,registros,pct_pasajeros_nulos
0,green_tripdata_2026-01,40272,13.44
1,green_tripdata_2026-02,37373,14.41
2,green_tripdata_2026-03,44208,15.14
3,green_tripdata_2026-04,44238,14.22
4,green_tripdata_2026-05,44921,12.85
5,green_tripdata_2026-06,44163,14.65
6,green_tripdata_2026-07,41252,15.60
7,green_tripdata_2026-08,40687,15.52
8,yellow_tripdata_2026-01,3724889,29.21
9,yellow_tripdata_2026-02,3399866,30.10


**Resultado.** Se encontraron siete problemas.

**Nulos en bloque.** El 25.98 % de los amarillos (7,716,688 viajes) no tiene `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge` ni `Airport_fee`, y todos esos viajes tienen `payment_type = 0`. En verdes pasa con el 14.47 %. Son registros sin la información del taxímetro, no faltantes aleatorios.

**Fechas fuera del periodo.** Hay viajes con inicio en 2001 y 2008. En total 146 amarillos y 98 verdes tienen una fecha que no corresponde al mes de su archivo.

**Duraciones imposibles.** 371,683 amarillos y 234 verdes terminan antes o en el mismo instante en que empiezan, y 258 viajes duran más de 24 horas.

**Distancias extremas.** El máximo de `trip_distance` es 328,522 millas en amarillos y 179,831 en verdes. Hay 1,295 viajes con más de 100 millas y casi un millón con distancia cero.

**Montos negativos.** 161,835 amarillos y 1,023 verdes tienen `total_amount` negativo, lo que corresponde a reversiones o ajustes.

**Códigos fuera del diccionario.** `RatecodeID = 99` y `VendorID` 6 y 7 no aparecen en todas las versiones del diccionario de la TLC.

**Columna casi vacía.** `ehail_fee` está 100 % nula en verdes y `request_source` lo está en más del 90 %.

**Decisión.** No se borra nada de los archivos originales. Los filtros de limpieza se aplicarán en las consultas de análisis (fecha dentro del mes del archivo, duración entre 1 y 180 minutos, distancia entre 0 y 100 millas y monto total positivo), y se documentan en cada consulta que los usa.

## 3.9 Qué significa consultar directamente un Parquet

Significa que DuckDB lee el archivo en su lugar, sin cargarlo antes a una tabla ni a memoria. Parquet guarda los datos por columnas y con estadísticas por grupo de filas, así que DuckDB lee solo las columnas que pide la consulta y salta los grupos que no cumplen los filtros.

**Por qué sirve con mucho volumen.** El conteo por metadatos de 30 millones de filas tardó milisegundos porque no leyó datos. El `SUMMARIZE` de todas las columnas de 29.7 millones de viajes tardó unos 16 segundos sin ocupar memoria más allá de lo que procesa en cada bloque, y no hubo un paso de importación que repetir cada vez que llega un archivo nuevo.